# Graphex Exact v1 • Google Colab • CUDA + resumable Google Drive

**Источник:** [Wishart CN100k k4](https://drive.google.com/drive/folders/1TCjdLFvJRNAb6tOJmoupcPcx057ng20X). Код: [feature/graphex-exact-v1](https://github.com/SemanticMap/semgraphex/tree/feature/graphex-exact-v1).

Ноутбук кодирует **сохранённые** `level_NNN/relations/*.npz` и `transition_NNN_NNN+1/figure_occurrences.jsonl`. Он не перезапускает Wishart, не меняет его контрольные точки и не заявляет точного восстановления отдельных исходных строк ConceptNet, которые могли быть агрегированы в CSR.

**GPU:** пакетная классификация рёбер W/S/I/R в PyTorch/CUDA. **CPU:** загрузка CSR, VF2, словарь, Хаффман, ZIP и точное декодирование; эти операции не объявляются GPU-ускоренными. При CUDA уровни запускаются последовательно, чтобы не переполнить VRAM; загрузка файлов распараллелена по CPU. При CPU-режиме допускается параллельная обработка уровней. Архивы и отчёты сохраняются в отдельную папку Google Drive. Повторный запуск проверяет контрольные суммы и пропускает только завершённые уровни.

> ВАЖНО: в текущем пилоте архив включает полный набор данных для обратимости; отрицательный выигрыш в байтах — корректный результат. Полная онлайн-MDL интеграция и глобальное иерархическое кодирование пока не реализованы.

## 1. Конфигурация эксперимента
По умолчанию — все 12 переходов (уровни 0–11). Можно выбрать, например, `LEVELS = [0, 4, 11]` для первого прохода.

In [ ]:
DRIVE_SOURCE_FOLDER_ID = "1TCjdLFvJRNAb6tOJmoupcPcx057ng20X"
GIT_REF = "feature/graphex-exact-v1"
LEVELS = list(range(12))  # уровни 0..11 имеют переход в следующий уровень
DEVICE = "auto"            # auto | cuda | cpu; auto приоритетно использует CUDA
GPU_BATCH_SIZE = 500_000
DOWNLOAD_WORKERS = 6       # параллельная загрузка файлов по CPU
CPU_LEVEL_WORKERS = 2      # используется только если DEVICE=cpu или CUDA недоступна
SCRATCH_ROOT = "/content/graphex_exact_v1"
RESULTS_FOLDER_NAME = "graphex_exact_v1_results"
RUN_ID = "wishart-cn100k-k4-graphex-exact-v1"
VERIFY_REMOTE_SHA256 = True

## 2. Preflight до установки зависимостей
Не предполагаем конкретную модель GPU или объём RAM. Большие вычисления выполняются на локальном SSD Colab.

In [ ]:
import os, shutil, sys, platform
from pathlib import Path
print("Python:", sys.version.split()[0], "CPU:", os.cpu_count())
print("RAM available (GiB):", round(int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 1024**2, 2))
print("Scratch free (GiB):", round(shutil.disk_usage("/content").free / 1024**3, 2))
if shutil.disk_usage("/content").free < 3 * 1024**3:
    raise RuntimeError("Для staging требуется не менее 3 GiB свободного места на /content.")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
ROOT = Path(SCRATCH_ROOT); ROOT.mkdir(parents=True, exist_ok=True)
assert len(set(LEVELS)) == len(LEVELS) and all(isinstance(k,int) and 0 <= k < 12 for k in LEVELS)

## 3. Код репозитория и зависимости
Версия Git фиксируется в итоговом отчёте; Torch не импортируется до завершения установки.

In [ ]:
import subprocess
REPO = ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                    "https://github.com/SemanticMap/semgraphex.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", GIT_REF, "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c",
                str(REPO / "requirements/constraints.txt"), "-e", str(REPO) + "[wishart]",
                "google-api-python-client", "google-auth-httplib2"], check=True)
print("Repository commit:", COMMIT)

## 4. CUDA и загрузка библиотек
CUDA будет использована **для классификации рёбер**, а VF2 и архивирование остаются CPU-задачами.

In [ ]:
import torch
import numpy as np
import scipy
cuda_available = torch.cuda.is_available()
if DEVICE == "cuda" and not cuda_available:
    raise RuntimeError("DEVICE=cuda, но Colab GPU недоступен: Runtime → Change runtime type → T4/L4/A100.")
actual_device = "cuda" if DEVICE == "cuda" or (DEVICE == "auto" and cuda_available) else "cpu"
print("Selected device:", actual_device)
if cuda_available:
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "VRAM GiB:", round(props.total_memory / 1024**3, 2))
print("torch", torch.__version__, "NumPy", np.__version__, "SciPy", scipy.__version__)

## 5. Google Drive API: чтение исходной папки и отдельная папка результатов
Аутентификация выполняется средствами Colab. Файлы скачиваются **на /content**, а не обрабатываются на смонтированном Google Drive.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive = build("drive", "v3", credentials=creds, cache_discovery=False)
FOLDER = "application/vnd.google-apps.folder"

def children(parent):
    found, page = [], None
    while True:
        response = drive.files().list(
            q=f"'{parent}' in parents and trashed=false",
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",
            pageSize=1000, pageToken=page, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        found.extend(response.get("files", []))
        page = response.get("nextPageToken")
        if not page: return found

def one_child(parent, name, kind=None):
    matches = [f for f in children(parent)
               if f["name"] == name and (kind is None or f["mimeType"] == kind)]
    if len(matches) != 1:
        raise RuntimeError(f"Ожидался ровно один {name!r} внутри {parent}, найдено {len(matches)}")
    return matches[0]

def ensure_folder(parent, name):
    matches = [f for f in children(parent) if f["name"] == name]
    if matches:
        if len(matches) != 1 or matches[0]["mimeType"] != FOLDER:
            raise RuntimeError("Имя папки результатов неоднозначно")
        return matches[0]["id"]
    return drive.files().create(body={"name":name,"mimeType":FOLDER,"parents":[parent]},
                                fields="id").execute()["id"]

source_meta = drive.files().get(fileId=DRIVE_SOURCE_FOLDER_ID,fields="id,name,mimeType").execute()
assert source_meta["mimeType"] == FOLDER, source_meta
result_folder_id = ensure_folder(DRIVE_SOURCE_FOLDER_ID, RESULTS_FOLDER_NAME)
run_folder_id = ensure_folder(result_folder_id, RUN_ID)
print("Source:", source_meta["name"], "results folder:", run_folder_id)

## 6. Параллельный staging с SHA/MD5
Загружаются только входные файлы выбранных уровней. После каждого уровня локальные файлы можно удалить. Если папка ещё синхронизируется, ноутбук сообщит, какой обязательный файл отсутствует.

In [ ]:
import hashlib, json, threading, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
_tls = threading.local()

def local_drive():
    if not hasattr(_tls, "drive"):
        _tls.drive = build("drive","v3",credentials=creds,cache_discovery=False)
    return _tls.drive

def sha256(path):
    h = hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(4*1024*1024),b""): h.update(block)
    return h.hexdigest()

def download_one(item, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and item.get("md5Checksum"):
        h = hashlib.md5()
        with dest.open("rb") as f:
            for part in iter(lambda:f.read(4*1024*1024),b""): h.update(part)
        if h.hexdigest() == item["md5Checksum"]: return {"path":str(dest),"sha256":sha256(dest)}
    part = dest.with_name(dest.name + ".part")
    try:
        with part.open("wb") as out:
            request = local_drive().files().get_media(fileId=item["id"])
            downloader = MediaIoBaseDownload(out, request, chunksize=8*1024*1024)
            done = False
            while not done: _,done = downloader.next_chunk(num_retries=5)
        if item.get("md5Checksum"):
            h = hashlib.md5()
            with part.open("rb") as f:
                for chunk in iter(lambda:f.read(4*1024*1024),b""): h.update(chunk)
            if h.hexdigest() != item["md5Checksum"]:
                raise RuntimeError("Drive MD5 mismatch for " + item["name"])
        part.replace(dest)
    finally:
        part.unlink(missing_ok=True)
    return {"path":str(dest),"sha256":sha256(dest)}

def tree_files(folder_id, relative=Path(".")):
    out=[]
    for item in children(folder_id):
        here=relative/item["name"]
        if item["mimeType"]==FOLDER: out.extend(tree_files(item["id"],here))
        else: out.append((item,here))
    return out

def prepare_level(level):
    stage=ROOT/"staging"/f"level_{level:03d}"
    source_level=one_child(DRIVE_SOURCE_FOLDER_ID,f"level_{level:03d}",FOLDER)
    transition=one_child(DRIVE_SOURCE_FOLDER_ID,f"transition_{level:03d}_{level+1:03d}",FOLDER)
    completion=one_child(DRIVE_SOURCE_FOLDER_ID,"COMPLETED")
    wanted=[(completion,Path("COMPLETED"))]
    wanted += [(item,Path(f"level_{level:03d}")/relative)
               for item,relative in tree_files(source_level["id"])
               if relative.name=="adjacency.npz" or relative.parts[0]=="relations"]
    wanted += [(item,Path(f"transition_{level:03d}_{level+1:03d}")/relative)
               for item,relative in tree_files(transition["id"])
               if relative.name=="figure_occurrences.jsonl"]
    required={"COMPLETED",f"level_{level:03d}/adjacency.npz",
              f"level_{level:03d}/relations/index.json",
              f"transition_{level:03d}_{level+1:03d}/figure_occurrences.jsonl"}
    got={str(relative) for _,relative in wanted}
    if not required.issubset(got):
        raise RuntimeError("Отсутствуют обязательные файлы: "+str(sorted(required-got)))
    if not any(len(relative.parts)==3 and relative.parts[1]=="relations"
               and relative.suffix==".npz" for _,relative in wanted):
        raise RuntimeError("Не найдены relations/*.npz")
    output={}
    with ThreadPoolExecutor(max_workers=min(DOWNLOAD_WORKERS, max(1,os.cpu_count() or 1))) as pool:
        futures={pool.submit(download_one,item,stage/relative):str(relative)
                 for item,relative in wanted}
        for future in as_completed(futures):
            output[futures[future]]=future.result()["sha256"]
    return stage,output

## 7. Запуск с checkpoint и RESUME
Единица checkpoint — завершённый уровень: ZIP, JSON-отчёт и `COMPLETED` загружаются на Drive именно в этом порядке. При повторном запуске готовый уровень пропускается только если совпадают версия кода, параметры и SHA-256 всех входных файлов.

In [ ]:
from googleapiclient.http import MediaFileUpload
import subprocess, traceback

def remote_json(file):
    import io
    stream=io.BytesIO()
    req=drive.files().get_media(fileId=file["id"])
    down=MediaIoBaseDownload(stream,req)
    done=False
    while not done: _,done=down.next_chunk(num_retries=5)
    return json.loads(stream.getvalue())

def upload(parent,path,name,mime):
    # Never overwrite an existing checkpoint file with the same name.
    existing=[x for x in children(parent) if x["name"]==name]
    if existing: raise RuntimeError("Output name already exists: "+name)
    media=MediaFileUpload(str(path),mimetype=mime,resumable=True,
                          chunksize=8*1024*1024)
    req=drive.files().create(body={"name":name,"parents":[parent]},
                              media_body=media,fields="id,name,size,md5Checksum")
    resp=None
    while resp is None: _,resp=req.next_chunk(num_retries=5)
    return resp

def encode_one(level):
    stage,fingerprints=prepare_level(level)
    params={"format":"graphex_exact_v1","git_commit":COMMIT,"level":level,
            "device":actual_device,"gpu_batch_size":GPU_BATCH_SIZE,
            "input_sha256":fingerprints}
    parent=ensure_folder(run_folder_id,f"level_{level:03d}")
    remote={item["name"]:item for item in children(parent)}
    if {"COMPLETED","report.json","graphex.zip"}.issubset(remote):
        saved=remote_json(remote["report.json"])
        if saved.get("run_params")==params and saved.get("archive_sha256"):
            print(f"[RESUME] level {level}: matching manifest found")
            # Download remote ZIP only when requested for independent SHA check.
            if VERIFY_REMOTE_SHA256:
                checked=ROOT/"verification"/f"level_{level:03d}.zip"
                download_one(remote["graphex.zip"],checked)
                if sha256(checked)!=saved["archive_sha256"]:
                    raise RuntimeError("Remote checkpoint SHA mismatch at level "+str(level))
                checked.unlink(missing_ok=True)
            shutil.rmtree(stage,ignore_errors=True)
            return saved
        raise RuntimeError("Existing COMPLETED level has incompatible config/code/input; use a new RUN_ID")
    if remote:
        raise RuntimeError(f"Partial remote checkpoint for level {level}: {sorted(remote)}. "
                           "Inspect it and select a fresh RUN_ID; no silent overwrite.")
    local=ROOT/"outputs"/f"level_{level:03d}"
    local.mkdir(parents=True,exist_ok=True)
    archive=local/"graphex.zip"
    cmd=[sys.executable,"-m","semmap_haken.graphex_offline",
         "--run",str(stage),"--level",str(level),"--output",str(archive),
         "--device",actual_device,"--gpu-batch-size",str(GPU_BATCH_SIZE)]
    print("[RUN]", " ".join(cmd))
    p=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True)
    if p.returncode:
        (local/"FAILED.txt").write_text(p.stderr[-12000:],encoding="utf-8")
        raise RuntimeError(f"level {level} failed: {p.stderr[-3000:]}")
    report=json.loads(p.stdout)
    report.update({"run_params":params,"archive_sha256":sha256(archive)})
    (local/"report.json").write_text(json.dumps(report,ensure_ascii=False,
                                               indent=2,sort_keys=True)+"\n",encoding="utf-8")
    # Upload data first; publish COMPLETED last.
    upload(parent,archive,"graphex.zip","application/zip")
    upload(parent,local/"report.json","report.json","application/json")
    completed=local/"COMPLETED"
    completed.write_text("complete\n",encoding="utf-8")
    upload(parent,completed,"COMPLETED","text/plain")
    shutil.rmtree(stage,ignore_errors=True)
    print("[DONE] level",level,"device",report["classification_device"],
          "saved bytes",report["net_saved_bytes"])
    return report

# Exact-coded levels share a GPU; process them sequentially to avoid VRAM contention.
# CPU-only mode can run across levels in parallel, but Drive API writes are
# coordinated per level and memory requirements can be large.
reports=[]
if actual_device=="cuda" or CPU_LEVEL_WORKERS<=1:
    for level in LEVELS:
        reports.append(encode_one(level))
else:
    # Each level handles its own disjoint Drive output folder.
    with ThreadPoolExecutor(max_workers=CPU_LEVEL_WORKERS) as pool:
        futures={pool.submit(encode_one,level):level for level in LEVELS}
        for future in as_completed(futures):
            reports.append(future.result())
reports.sort(key=lambda row:row["level"])
print("Completed levels:",[item["level"] for item in reports])

## 8. Реальные биты, обратимость, W/S/I/R
Суммы здесь относятся к **отдельным архивам уровней**. Не интерпретируйте их как длину единого иерархического кода исходного ConceptNet.

In [ ]:
import pandas as pd
table=pd.DataFrame([{
    "level":r["level"],"device":r["classification_device"],
    "edges":r["edge_records"],"W":r["partition"].get("W",0),
    "S":r["partition"].get("S",0),"I":r["partition"].get("I",0),
    "R":r["partition"].get("R",0),
    "W_shapes":r["shapes"],"archive_bytes":r["archive_bytes"],
    "baseline_bytes":r["baseline_bytes"],
    "saved_bytes":r["net_saved_bytes"],
    "symbol_bits":r["stream_bits"],"exact":r["roundtrip_exact"]
} for r in reports])
display(table)
assert table["exact"].all()
assert (table[["W","S","I","R"]].sum(axis=1)==table["edges"]).all()
print("All tested levels: exact CSR-record roundtrip; component conservation OK.")
print("Result folder: https://drive.google.com/drive/folders/"+run_folder_id)

## Ограничения производительности и проверки

- CUDA ускоряет именно проверяемую пакетную классификацию; VF2 и ZIP остаются CPU-bound. Запуск на GPU не гарантирует ускорения всей программы, особенно на небольших уровнях.
- Размер W-словаря и реальный выигрыш зависят от формы подграфов; результат может быть отрицательным.
- Если папка ещё не содержит `COMPLETED` или требуемые CSR-файлы, код сообщит об этом и не создаст фиктивный отчёт.
- Честная проверка после RESUME выполняется сравнением параметров, SHA всех входов и, по умолчанию, скачанного архива.
- Для проверки полного исходного ConceptNet необходимы неагрегированные идентификаторы строк и отдельный декодер исходного уровня; текущая гарантия относится к CSR-слоям.